# Emotion CNN — Training

Trains the TinyVGG-style 1D CNN with a cost-sensitive loss on the collected sensor data.
Uses the `emorec` package for all data loading, windowing, and model definitions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from emorec import build_dataset, EmotionCNN, CostSensitiveLoss, PENALTY_GRID

## 1. Load Dataset

In [ ]:
DATA_DIR = 'EmoRecData'

# Files held out for final evaluation — not seen during training
HOLDOUT_FILES = [
    'adi_7_5min_stress.csv', 'adi_7_5min_focus.csv',
    'adi_7_5min_baseline.csv', 'adi_7_5min_distract.csv',
    'louis_7_5min_distract.csv', 'louis_7_5min_baseline.csv',
    'louis_7_5min_stress.csv', 'louis_7_5min_focus.csv',
    'emmanuel_7_5min_baseline.csv', 'emmanuel_7_5min_distract.csv',
    'emmanuel_7_5min_stress.csv', 'emmanuel_7_5min_focus.csv',
    'focus_test.csv', 'adi_focused.csv', 'louis_focused.csv', 'louis_stressed.csv',
]

X, y_int, label_encoder = build_dataset(DATA_DIR, ignore_files=HOLDOUT_FILES)
print(f'\nClasses: {label_encoder.classes_}')

## 2. Train / Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_int, test_size=0.2, random_state=42, stratify=y_int
)
print(f'Train: {X_train.shape[0]} samples  |  Val: {X_val.shape[0]} samples')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Transpose to (batch, channels, time) for Conv1d
X_train_t = torch.FloatTensor(X_train).transpose(1, 2)
X_val_t   = torch.FloatTensor(X_val).transpose(1, 2)
y_train_t = torch.LongTensor(y_train)
y_val_t   = torch.LongTensor(y_val)

BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=BATCH_SIZE, shuffle=False)

print(f'Input shape per batch: {next(iter(train_loader))[0].shape}')

## 3. Model & Optimiser

In [ ]:
model     = EmotionCNN(input_channels=14, hidden_units=10, num_classes=4).to(device)
criterion = CostSensitiveLoss(cost_matrix=PENALTY_GRID)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Training Loop

In [ ]:
EPOCHS = 200
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0
SAVE_PATH = 'Ml-Models/emotion_cnn_best.pth'

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bX)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        correct += (out.argmax(1) == by).sum().item()
        total += by.size(0)
    train_losses.append(running_loss / len(train_loader))
    train_accs.append(correct / total)

    # --- Validate ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for bX, by in val_loader:
            bX, by = bX.to(device), by.to(device)
            out = model(bX)
            v_loss += criterion(out, by).item()
            v_correct += (out.argmax(1) == by).sum().item()
            v_total += by.size(0)
    val_losses.append(v_loss / len(val_loader))
    val_acc = v_correct / v_total
    val_accs.append(val_acc)
    scheduler.step(v_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)

    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'train_loss={train_losses[-1]:.4f}  train_acc={train_accs[-1]:.3f}  '
              f'val_loss={val_losses[-1]:.4f}  val_acc={val_accs[-1]:.3f}')

print(f'\nBest val accuracy: {best_val_acc:.3f}  →  {SAVE_PATH}')

## 5. Training Curves

In [ ]:
epochs = range(1, len(train_losses) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, train_losses, label='Train')
ax1.plot(epochs, val_losses,   label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.4)

ax2.plot(epochs, train_accs, label='Train')
ax2.plot(epochs, val_accs,   label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.4)

plt.tight_layout()
plt.show()

## 6. Evaluation

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for bX, by in val_loader:
        preds = model(bX.to(device)).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(by.numpy())

print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_).plot(cmap='Blues', ax=ax)
ax.set_title(f'Validation Confusion Matrix  (best val acc = {best_val_acc:.1%})')
plt.tight_layout()
plt.show()